# Residual recalibration — T8, one named constant per circuit

After T1-T7, the leftover per calibration circuit is c_circuit = median(real 2026 lap
time) − median(model-implied lap time), which for a circuit with a real fit isolates the
transform's intercept error. The pooled c_global is applied to circuits with no 2026 data.
The held-out sim-to-real error on rounds 7+ is the headline number, and it is computed in
`validate.py` (the only reader of rounds 7+, E-14) — never here, or the residual would
validate itself. Logic lives in `senna.physics.synth.residual`.

In [ ]:
import argparse
import asyncio
import os

import numpy as np
import pandas as pd

from senna import config
from senna.data.db import database_url, engine_for
from senna.physics.fits import frames as fit_frames
from senna.physics.synth import plots

synth_cfg = config.load("synth")
physics_2026 = config.load("physics_2026")
CALIB = config.get(config.load("data_split"), "calibration_rounds")

# the battery needs Postgres + the FastF1 cache; export DATABASE_URL first
assert os.environ.get("DATABASE_URL"), "export DATABASE_URL (see .env.example) before running"

In [ ]:
from senna.physics.synth import run

# a full run computes c_circuit on every calibration circuit with a real 2026 fit
args = argparse.Namespace(only=None, circuit=None, transforms_out=None, physics_2026_out=None)
transforms = asyncio.run(run.run_battery(args))
{slug: c["t8_residual"] for slug, c in transforms["circuits"].items() if "t8_residual" in c}